# Lyric Engine — Colab Training Notebook

**Runtime:** GPU (T4 free / A100 Colab Pro)  
**What this does:** Fine-tunes a base LLM on genre-tagged lyrics with phoneme annotations.

Stages:
1. Setup & install
2. Mount Drive (checkpoint storage)
3. Data collection via Genius API
4. Phoneme annotation pipeline
5. Stage 1: General music SFT
6. Stage 2: Per-genre LoRA adapters
7. Test generation


In [ ]:
# ── 1. Check GPU ──────────────────────────────────────────────────────────────
import torch
print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE — switch runtime to GPU"}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB' if torch.cuda.is_available() else '')
!nvidia-smi

In [ ]:
# ── 2. Install dependencies ───────────────────────────────────────────────────
!pip install -q transformers peft accelerate bitsandbytes trl datasets \
    lyricsgenius pronouncing sentence-transformers textblob \
    fastapi uvicorn rich typer
print('Done.')

In [ ]:
# ── 3. Clone repo & mount Drive ───────────────────────────────────────────────
!git clone https://github.com/SMXFREEZE/lyric-engine
%cd lyric-engine

from google.colab import drive
drive.mount('/content/drive')

import os
CHECKPOINT_DIR = '/content/drive/MyDrive/lyric-engine-checkpoints'
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs('data/raw', exist_ok=True)
print(f'Checkpoints will save to: {CHECKPOINT_DIR}')

In [ ]:
# ── 4. Config ─────────────────────────────────────────────────────────────────
import os

# Paste your tokens here
GENIUS_TOKEN = ''          # get from genius.com/api-clients
HF_TOKEN     = ''          # get from huggingface.co/settings/tokens

os.environ['GENIUS_TOKEN'] = GENIUS_TOKEN
os.environ['HUGGING_FACE_HUB_TOKEN'] = HF_TOKEN

# Model choice — change to Llama if you have HF access
BASE_MODEL   = 'mistralai/Mistral-7B-v0.1'   # free, no approval needed
# BASE_MODEL = 'meta-llama/Llama-3.1-8B'     # needs HF approval

GENRE        = 'trap'      # which genre adapter to train in Stage 2
MAX_SONGS    = 200         # songs per artist (reduce if hitting rate limits)
BATCH_SIZE   = 2           # reduce to 1 if OOM
GRAD_ACCUM   = 16          # effective batch = BATCH_SIZE * GRAD_ACCUM

print(f'Base model : {BASE_MODEL}')
print(f'Genre      : {GENRE}')
print(f'Eff. batch : {BATCH_SIZE * GRAD_ACCUM}')

In [ ]:
# ── 5. Scrape lyrics ──────────────────────────────────────────────────────────
import sys
sys.path.insert(0, '.')

from src.data.scraper import run_scrape

# Scrape only the target genre to save time
records = run_scrape(
    genres=[GENRE],
    max_songs_per_artist=MAX_SONGS,
    out_dir='data/raw',
    token=GENIUS_TOKEN,
)
print(f'Collected {len(records)} songs')

In [ ]:
# ── 6. Annotate phonemes + compute style vectors ──────────────────────────────
import json
from src.data.style_extractor import build_style_vectors

print('Building artist style vectors...')
style_vectors = build_style_vectors(
    jsonl_path=f'data/raw/{GENRE}.jsonl',
    out_dir='data/style_vectors',
    min_songs=3,
)
print(f'Style vectors built for {len(style_vectors)} artists')

In [ ]:
# ── 7. Verify data pipeline ───────────────────────────────────────────────────
from src.data.phoneme_annotator import annotate_line
from src.data.rhyme_labeler import detect_scheme
from src.data.valence_scorer import score_lyrics

with open(f'data/raw/{GENRE}.jsonl') as f:
    sample = json.loads(f.readline())

lines = sample['lyrics'].splitlines()[:8]
print(f"Artist: {sample['artist']} | Title: {sample['title']}")
print()

scheme = detect_scheme(lines)
print(f"Rhyme scheme: {scheme['scheme_str']} ({scheme['scheme_type']})")
print(f"Rhyme density: {scheme['rhyme_density']}")
print()

for em in score_lyrics('\n'.join(lines)):
    ann = annotate_line(em.text)
    print(f"  [{em.valence:+.2f}v {em.arousal:.2f}a] [{ann.total_syllables}syl] {em.text[:60]}")

In [ ]:
# ── 8. Load base model ────────────────────────────────────────────────────────
from src.model.lyrics_model import load_base_model, apply_lora, LyricsModel

device = 'cuda'
use_4bit = True   # essential for T4 (16GB VRAM)

base, tokenizer = load_base_model(BASE_MODEL, use_4bit=use_4bit, device=device)
base = apply_lora(base, rank=16, alpha=32)
model = LyricsModel(base, d_model=base.config.hidden_size)

base.print_trainable_parameters()
print(f'Model on: {next(model.parameters()).device}')

In [ ]:
# ── 9. Train — Stage 2 genre LoRA ─────────────────────────────────────────────
from src.training.sft import train_sft

train_sft(
    stage=2,
    genre=GENRE,
    data_path=f'data/raw/{GENRE}.jsonl',
    base_model=BASE_MODEL,
    output_dir=CHECKPOINT_DIR,
    batch_size=BATCH_SIZE,
    grad_accum_steps=GRAD_ACCUM,
    epochs=1,
    lr=2e-4,
    lora_rank=16,
    use_4bit=use_4bit,
    style_vec_dir='data/style_vectors',
)

In [ ]:
# ── 10. Test generation with trained adapter ───────────────────────────────────
from transformers import AutoModelForCausalLM, AutoTokenizer as HFTok
from peft import PeftModel
from src.inference.engine import LyricsEngine, SongMemory

ckpt_path = f'{CHECKPOINT_DIR}/genre_{GENRE}/epoch_1'

tok = HFTok.from_pretrained(ckpt_path)
tok.pad_token = tok.eos_token

base_mdl = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    load_in_4bit=True,
    device_map='auto',
)
trained_mdl = PeftModel.from_pretrained(base_mdl, ckpt_path)

engine = LyricsEngine(trained_mdl, tok, device=device, beam_size=8)
memory = SongMemory(genre=GENRE, rhyme_scheme='AABB', target_syllables=10)
memory.sections.append(('[BUILD]', 'VERSE'))

print(f'\n=== Generated {GENRE.upper()} verse ===')
verse = engine.generate_verse(memory, num_lines=8, section='VERSE', arc_token='[BUILD]')
for line in verse:
    print(f'  {line}')

In [ ]:
# ── 11. Save to Drive ─────────────────────────────────────────────────────────
print(f'Checkpoint saved at: {CHECKPOINT_DIR}/genre_{GENRE}/epoch_1')
print('Files in checkpoint:')
import os
for f in os.listdir(f'{CHECKPOINT_DIR}/genre_{GENRE}/epoch_1'):
    size = os.path.getsize(f'{CHECKPOINT_DIR}/genre_{GENRE}/epoch_1/{f}')
    print(f'  {f:40s} {size/1e6:.1f} MB')